# 10. Personal Assistant Subagents — Delegate by role

A personal assistant often handles mixed requests: calendar work, email drafting, research, and follow-up planning. This example uses role-based subagents to keep those responsibilities separate.

**Learning goals**
- Define subagent roles around user-facing responsibilities.
- Route requests to the right role.
- Merge multiple role outputs into one assistant response.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

## 10.1 Define roles

Good subagent design starts with role boundaries. Each role should have a clear job, input shape, and output expectation.


In [ ]:
subagents = {
    "calendar": {"tools": ["check_availability"], "risk": "approval"},
    "email": {"tools": ["draft_reply"], "risk": "review"},
    "research": {"tools": ["search_notes"], "risk": "allow"},
}

subagents

## 10.2 Route requests

Routing turns a user request into an ownership decision. Deterministic routing is enough for this small example and keeps the behavior easy to inspect.


In [ ]:
def route_request(text: str) -> str:
    lowered = text.lower()
    if "meeting" in lowered or "schedule" in lowered:
        return "calendar"
    if "email" in lowered or "email" in lowered:
        return "email"
    return "research"

route_request("Check tomorrow schedule and prepare an email draft")

## 10.3 Decompose compound requests

Real requests often contain more than one job. Decomposition lets the assistant delegate each part without losing the overall user intent.


In [ ]:
def plan_tasks(text: str) -> list[dict]:
    tasks = []
    for name in subagents:
        if name == route_request(text) or name in text.lower():
            tasks.append({"subagent": name, "input": text})
    return tasks or [{"subagent": "research", "input": text}]

plan_tasks("Check calendar and draft email")

## 10.4 Merge fan-in results

The final response should read like one assistant, not a pile of subagent logs. Fan-in synthesis combines role outputs into a useful summary.


In [ ]:
results = [
    {"subagent": "calendar", "result": "Tuesday at 3 PM is available"},
    {"subagent": "email", "result": "Drafted a meeting proposal email"},
]

summary = "\n".join(f"- {r['subagent']}: {r['result']}" for r in results)
print(summary)

---

## Summary

| Item | Content |
|---|---|
| **Covered** | supervisor routing, role-specific subagents, tool scopes, and fan-in summaries |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`multi-agent-middleware.md`](../../docs/langchain/multi-agent/subagents-personal-assistant.md)
- [`subagents.md`](../../docs/deepagents/subagents.md)
- [`async-subagents.md`](../../docs/deepagents/async-subagents.md)
